# Asia GARP Research v0.2.1 — Google Colab
AI Quant v5.12 모의계좌와 분리된 **연구 전용** 스크리너입니다. 실제 주문 기능이 없습니다.
프로젝트 ZIP을 업로드한 다음 공개 데이터 조회를 실행합니다.

기본 12개는 4개 산업군을 균형 추출합니다. 성공하면 `--max-symbols 24`로 확장하세요.


In [ ]:
%pip -q install "yfinance>=0.2.50,<2" "pandas>=2.2,<3" "numpy>=2.0,<3" "pytest>=8,<10"


In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import os
print("새 연구용 ZIP을 업로드해 주세요. 비밀키나 계좌 state는 올리지 마세요.")
uploaded = files.upload()
assert len(uploaded)==1, "프로젝트 ZIP 파일 하나만 선택해 주세요"
with ZipFile(BytesIO(next(iter(uploaded.values())))) as z:
    z.extractall("/content/asia_garp_workspace")
roots = list(Path("/content/asia_garp_workspace").rglob("requirements.txt"))
assert len(roots)==1, "프로젝트 루트가 하나 있어야 합니다"
os.chdir(str(roots[0].parent))
print("프로젝트 경로:",os.getcwd())


In [ ]:
!python -m pytest -q tests
!python -m src.screen --max-symbols 24 --output outputs


In [ ]:
import pandas as pd
from IPython.display import display, Markdown
from zipfile import ZipFile
from pathlib import Path
out=Path("outputs")
if (out/"03_research_report.md").exists():
    display(Markdown((out/"03_research_report.md").read_text(encoding="utf-8")))
    display(pd.read_csv(out/"02_research_shortlist.csv").head(20))
    display(pd.read_csv(out/"05_data_quality_review.csv").head(20))
    with ZipFile("asia_garp_research_outputs.zip","w") as z:
        for f in out.iterdir(): z.write(f,f.name)
    files.download("asia_garp_research_outputs.zip")
else:
    print("데이터 수집에 실패했습니다. 이전 셀의 오류 로그부터 확인하세요.")
